In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# ???? ???????
?? ???? ??????? ?? ?????????? ??????????? ?????????? ???????????? ???????? xgboost, lightgbm ? catboost.

?? ????? ???????????? ??????? "Bank Marketing". ? ???????? "Bank Marketing" ?????? ?????????? ??????, ????????? ? ?????????????? ?????????? ?????. ??? ????????? ????? ???????, ??????? ????? ?????????????? ? ???????? "Bank Marketing":

1. **age**: ??????? ???????.

2. **job**: ??? ?????? ???????.

3. **marital**: ???????? ????????? ???????.

4. **education**: ??????????? ???????.

5. **default**: ???? ?? ? ??????? ????????? ?????? (?????????????).

6. **balance**: ?????? ?? ????? ???????.

7. **housing**: ????? ?? ?????? ??????? (????? ????????????).

8. **loan**: ????? ?? ?????? ?????? ????.

9. **contact**: ?????? ????? ? ????????.

10. **day**: ???? ??????, ????? ??? ???????? ????????? ???????.

11. **month**: ????? ????, ????? ??? ???????? ????????? ???????.

12. **duration**: ????????????????? ?????????? ???????? ? ????????.

13. **campaign**: ?????????? ?????????, ??????????? ?? ????? ?????? ????????.

14. **pdays**: ?????????? ????, ????????? ? ??????? ?????????? ???????? ?? ?????????? ????????.

15. **previous**: ?????????? ?????????, ??????????? ????? ??????? ?????????.

16. **poutcome**: ????????? ?????????? ????????????? ????????.

17. **target_variable**: ??????? ??????????, ??????? ?????????? ???????????. ????????, ??? ????? ???? ?????????? ? ???, ???? ?? ?????? ??????????? (1) ??? ??? (0) ?? ???????????? ??????????? ?????.

???????? ????????, ??? ??????????? ???????? ??????? ? ?? ???? ?????? ????? ??????????? ? ??????????? ?? ??????????? ???????? "Bank Marketing", ??????? ????????????? ????????? ???????? ??????? ? ????? ?????????? ?????? ??????.

In [ ]:
!pip install pandas scikit-learn xgboost lightgbm catboost hyperopt optuna bayesian-optimization

In [ ]:
# ?????????? ?????? ??????????
import pandas as pd
import time

# ?????? ??????
from sklearn.metrics import f1_score

# ?????? ?????????????? ??????
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ?????????? ???????????? ????????
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# ???????? ?????? ?? CSV ?????
url = 'https://datahub.io/machine-learning/bank-marketing/r/bank-marketing.csv'
column_names = ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 
                'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 
                'poutcome', 'target_variable']

data = pd.read_csv(url)
data.columns = column_names
data.head()

# ?????????? ?? ???????? (X) ? ??????? ?????????? (y)
X = data.drop('target_variable', axis=1)
y = data['target_variable'].apply(lambda x: x - 1)


# ?????????? ?? ????????? ? ???????? ?????? ??????
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ??????????? ????????????? ??? ?????????????? ? ???????? ?????????
categorical_features = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
categorical_transformer = OneHotEncoder(handle_unknown='ignore')
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns
numerical_transformer = StandardScaler()

# ??????????? ??????????????
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', numerical_transformer, numerical_features)
    ])

# ???????? ????????? ??? ?????????????
preprocessing_pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# ?????????? ????????????? ? ??????
X_train = preprocessing_pipeline.fit_transform(X_train)
X_test = preprocessing_pipeline.transform(X_test)

## XGBoost

XGBoost (eXtreme Gradient Boosting) - ??? ?????????? ???????????? ????????, ??????????????? ??? ??????? ????? ????????????? ? ?????????. ??? ???????? ?? ???? ???????????? ????????, ??????? ?????? ???????? ?????? ??????? (?????? ???????? ???????) ? ?????????? ?? ??? ????????? ????? ??????? ? ?????????? ??????.

???????? ????????? XGBoost:

1. ????????????? ??????:
   - ???????????????? ?????? ? ???????? ?????????????? (????????, ??????? ????????? ??????? ?????????? ??? ?????? ????????? ??? ?????????? ????????? ?????? ??? ?????? ?????????????).

2. ?????????? ??????? ??????? (???????? ???????):
   - ?????? ?????? ???????? ????????. ?? ?????? ????? ??????????? ????? ?????? ? ?????? ????????, ?????????? ????? ?????????? ????????.
   - ??? ?????????? ?????? ???????????? ???????? ???????????????, ????? ??? ???????? ????? ??? ??????????? ????????, ??? ??????????? ?????????? ????????? ?? ?????? ???? ??????.
   - ??????? ???????? ? ???????????? ?? ?? ??????? ??? ??????? ???????????, ????? ???????? ????????????.

3. ?????????? ?????????? ? ?????????? ????????????:
   - ????? ?????????? ??????? ?????? ??????????? ????????? ?????? ????? ?????????????? ?????? ? ????????? ?????????? ??????? ??????????.
   - ???????????? ?????? ??????????? ????? ?????????? ?????????? ?????? ???????????? ?????? ??????. ???? ???????????? ????????? ???????? (learning rate), ??????? ???????????? ??????? ??????? ?????? ?? ???????? ???????????? ??????.

4. ????????????? ? ?????????????? ????????????:
   - ?????????????? ????????? ????????????? ? XGBoost ???????? ????????????? ???????????? ? ???????? ?????????? ??????????? ??????.
   - XGBoost ?????????? ????????? ??????? ?????????????, ????? ??? L1- ? L2-????????????? (????? ????????? ??? ????????????? ????? ? ????), ??????? ???????? ?????????????? ????????? ?????? ? ????????????? ????????????. ??? ?????? ????????? ?????? ? ??????? ??????, ??????? ??????? ?? ????? ??????.

6. ??????? ??????:
   - XGBoost ?????????? ?????????? ??????? ??????, ??????? ???????? ? ???? ????????? ??????? ?????? ? ??????????? ?? ???????? ??????? ??????????. ????????, ??? ?????? ????????????? ? ????? ???????? ????? ?????????????? ?????-???????? ($L(y, \hat{y}) = -y \log(\hat{y}) - (1-y) \log(1-\hat{y})$), ? ??? ?????? ????????? - ?????????????????? ?????? $(\frac{1}{n}\sum (y - \hat{y})^2)$.
   
7. ??????????????? ????????:
   - ????????? XGBoost ?????? ???????? ?? ?????????? ????????, ???????????? ??????? ?????? ????????????, ????? ???????? ???????? ???????????? ??????. ?????? ?????????? ???????? ???? ???????? ? ?????? ?? ?????, ???????????? ?? ?????? ?????? ? ??????????.

8. ???????????????:
    - ????? ???????? ?????? XGBoost ????? ???????????? ??? ??????????????? ?? ????? ??????. ?????? ????????? ??????? ???????? ? ?????????? ???????? ??? ????? ????????????? ??? ?????????.

??? ??????????? ?????? XGBoost ?????? ???????????? ????????? ???????? ?????????:

1. **max_depth**: ??????? ??????. ???? ???????? ????????????, ????????? ???????? ????? ???? ?????? ?????? ? ????????. ??????? ??????? ????? ???????? ? ????????????, ??????? ??? ??????? ??????????? ?????????.

2. **learning_rate**: ???????? ????????. ???? ???????? ?????????? ????? ??????? ?????? ? ????????. ????? ?????? ???????? learning_rate ??????? ???????? ????? ???????? ??? ?????????? ??????? ??????????????????, ?? ????? ???????? ?????????? ??????????? ??????.

3. **n_estimators**: ?????????? ???????? ? ????????. ???? ???????? ?????????, ??????? ???????? ??????? ?????????. ??????? ?????????? ???????? ????? ???????? ?????????????????? ??????, ?? ??????? ??????? ????? ????? ???????? ? ????????????.

4. **subsample**: ???? ????????? ????????, ???????????? ??? ?????????? ??????? ??????. ???? ???????? ???????????? ????????? ?????????? ?????? ??? ??????? ??????. ???????? ?????? 1.0 ????????? ????????? ???????????? ? ???????? ???????????? ??????.

5. **colsample_bytree**: ???? ?????????, ???????????? ??? ?????????? ??????? ??????. ???? ???????? ???????????? ????????? ?????????? ????????? ??? ??????? ??????. ???????? ?????? 1.0 ???????? ????????? ???????????? ? ???????? ???????????? ????????.

6. **colsample_bylevel**: ???? ?????????, ???????????? ??? ?????????? ??????? ?????? ?? ?????? ??????. ???? ???????? ???????????? ????????? ?????????? ????????? ??? ??????? ??????. ???????? ?????? 1.0 ???????? ????????? ???????????? ? ???????? ???????????? ????????.

7. **colsample_bynode**: ???? ?????????, ???????????? ??? ?????????? ??????? ?????? ? ?????? ???? ?????????. ???? ???????? ???????????? ????????? ?????????? ????????? ??? ??????? ??????. ???????? ?????? 1.0 ???????? ????????? ???????????? ? ???????? ???????????? ????????.

8. **gamma**: ??????????? ?????????? ??????? ??????, ??????????? ??? ?????????? ??????????? ? ??????. ???? ???????? ???????????? ????????????? ??????. ??????? ???????? gamma ???????? ????????????? ???????????? ?? ???? ?????????? ?????? ??? ?????????? ???????????.

9. **reg_alpha**: ???????? L1 ?????????????. ?? ???????? ?????????????? ????????? ?????? ? ????????????? ???????????? ????? ?????????? ??????? ?? ??????? ???????? ?????.

10. **reg_lambda**: ???????? L2 ?????????????. ?? ???????? ?????????????? ????????? ?????? ? ????????????? ???????????? ????? ?????????? ??????? ?? ??????? ???????? ?????.

In [ ]:
# ?????????? ? ?????? ???????????
from sklearn.model_selection import GridSearchCV

# ??????????? ?????? XGBoost
model = xgb.XGBClassifier()

# ??????????? ????? ?????????? ??? ???????????
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.1],
    'n_estimators': [30]
}

# ???????? ??????? GridSearchCV ??? ??????????? ??????????
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='f1')

# ???????? ?????? ? ?????????????? GridSearchCV
start_time = time.time()
grid_search.fit(X_train, y_train)
xgb_gridsearch_execution_time = time.time() - start_time
print("????? ?????? xgb_gridsearch_execution_time:", xgb_gridsearch_execution_time)

# ?????? ????????? ? ???????? ???????
print("?????? ?????????:", grid_search.best_params_)
print("?????? F1:", grid_search.best_score_)

# ?????? ???????? ?????? ?? ???????? ?????? ??????
model = xgb.XGBClassifier(**grid_search.best_params_)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
xgb_gridsearch_f1 = f1_score(y_pred, y_test)
print("F1 ?? ???????? ?????? ??????:", xgb_gridsearch_f1)

In [ ]:
from hyperopt import fmin, tpe, hp, Trials

# ??????????? ??????? ??? ???????????
def objective(params):
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return -f1_score(y_test, y_pred)  # ??????? ???????, ????????????? F1-????

# ??????????? ???????????? ?????? ??????????
space = {
    'max_depth': hp.choice('max_depth', [3, 4, 5]),
    'learning_rate': hp.choice('learning_rate', [0.1, 0.01, 0.001]),
    'n_estimators': hp.choice('n_estimators', [30, 50, 100])
}

# ?????????? ???????????
trials = Trials()
start_time = time.time()
best = fmin(objective, space, algo=tpe.suggest, max_evals=10, trials=trials)
xgb_hyperopt_execution_time = time.time() - start_time
print("????? ?????? xgb_hyperopt_execution_time:", xgb_hyperopt_execution_time)

# ?????? ????????? ? ???????? ???????
best_params = {
    'max_depth': [3, 4, 5][best['max_depth']],
    'learning_rate': [0.1, 0.01, 0.001][best['learning_rate']],
    'n_estimators': [30, 50, 100][best['n_estimators']]
}
# ?????? ????????? ? ???????? ???????
best_score = -trials.best_trial['result']['loss']


print("?????? ?????????:", best_params)
print("?????? F1-????:", best_score)

# ?????? ???????? ?????? ?? ???????? ?????? ??????
model = xgb.XGBClassifier(**best_params)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
xgb_hyperopt_f1 = f1_score(y_pred, y_test)
print("F1 ?? ???????? ?????? ??????:", xgb_hyperopt_f1)

In [ ]:
import optuna

# ??????????? ??????? ??? ???????????
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1),
        'n_estimators': trial.suggest_categorical('n_estimators', [30, 50, 100])
    }
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    
    return f1  # ??????? ???????, ????????????? F1-????

# ???????? ??????? Study ??? ???????????
study = optuna.create_study(direction='maximize')
start_time = time.time()
study.optimize(objective, n_trials=10)
xgb_optuna_execution_time = time.time() - start_time
print("????? ?????? xgb_optuna_execution_time:", xgb_optuna_execution_time)

# ?????? ????????? ? ???????? ???????
best_params = study.best_params
best_score = study.best_value

print("?????? ?????????:", best_params)
print("?????? F1-????:", best_score)

# ?????? ???????? ?????? ?? ???????? ?????? ??????
model = xgb.XGBClassifier(**best_params)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
xgb_optuna_f1 = f1_score(y_pred, y_test)
print("F1 ?? ???????? ?????? ??????:", xgb_optuna_f1)

In [ ]:
from bayes_opt import BayesianOptimization
from sklearn.metrics import f1_score

# ??????????? ??????? ??? ???????????
def objective(max_depth, learning_rate, n_estimators):
    params = {
        'max_depth': int(max_depth),
        'learning_rate': learning_rate,
        'n_estimators': int(n_estimators)
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    
    return f1

# ??????????? ????????? ???????? ?????????? ??? ???????????
pbounds = {
    'max_depth': (3, 5),
    'learning_rate': (0.001, 0.1),
    'n_estimators': (30, 100)
}

# ???????? ??????? BayesianOptimization ??? ???????????
optimizer = BayesianOptimization(
    f=objective,
    pbounds=pbounds,
    random_state=42,
    verbose=2
)

# ?????? ???????????
start_time = time.time()
optimizer.maximize(init_points=2, n_iter=10)
xgb_bayes_opt_execution_time = time.time() - start_time
print("????? ?????? xgb_bayes_opt_execution_time:", xgb_bayes_opt_execution_time)

# ?????? ????????? ? ???????? ???????
best_params = optimizer.max['params']
best_f1 = optimizer.max['target']

print("?????? ?????????:", best_params)
print("?????? F1-????:", best_f1)

# ?????? ???????? ?????? ?? ???????? ?????? ?????? ? ??????? ???????????
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['max_depth'] = int(best_params['max_depth'])
best_model = xgb.XGBClassifier(**best_params)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
xgb_bayes_opt_f1 = f1_score(y_test, y_pred)

print("F1-???? ?? ???????? ?????? ??????:", xgb_bayes_opt_f1)


## LightGBM

???????? ????????? LightGBM:

1. ????????????? ??????:
   - ???????????????? ?????? ? ???????? ?????????????? (????????, ??????? ????????? ??????? ?????????? ??? ?????? ????????? ??? ?????????? ????????? ?????? ??? ?????? ?????????????).

2. ?????????? ??????? ??????? (???????? ???????):
   - LightGBM ?????????? ???????? ???????????? ???????? ??? ????????? ?????????.
   - ??????? ???????? ????????, ?????????? XGBoost. ?? ?????? ????? ??????????? ????? ?????? ? ?????? ????????, ?????????? ????? ?????????? ????????.
   - LightGBM ?????????? ???????????????? ?????? ????????? ???????????? ????????, ??????? ???????? ?? ?????? ???????? ?? ????????????. ??? ????????? ??????????? ???????? ??????? ?????????? ????????.

3. ?????????? ?????????? ? ?????????? ????????????:
   - ????? ?????????? ??????? ?????? ??????????? ????????? ?????? ????? ?????????????? ?????? ? ????????? ?????????? ??????? ??????????.
   - ???????????? ?????? ??????????? ????? ?????????? ?????????? ?????? ???????????? ?????? ??????, ?????????? XGBoost.

4. ????????????? ? ?????????????? ????????????:
   - LightGBM ????? ?????????? ????????? ??????? ????????????? ??? ?????????????? ???????????? ??????.
   - ?? ???????????? L1-????????????? (????????????? ?????) ? L2-????????????? (????????????? ????), ??????? ????????? ?????? ? ??????? ??????, ?????????? XGBoost.

6. ??????? ??????:
   - LightGBM ???????????? ????????? ??????? ??????, ? ??????????? ?? ???? ?????? (????????????? ??? ?????????). ??? ?????? ???????? ????????????? ????? ???????????? ????????????? ??????? ??????, ? ??? ?????? ????????? - ?????????????????? ?????? (MSE), ?????????? XGBoost.

7. ??????????????? ????????:
   - LightGBM ????? ?????? ???????? ?? ?????????? ????????, ? ???????????? ??????? ?????? ???????????? ??? ????????? ????????? ???????????? ??????.

8. ???????????????:
   - ????? ???????? ?????? LightGBM ????? ???????????? ??? ??????????????? ?? ????? ??????, ?????????? XGBoost. ?????? ????????? ??????? ???????? ? ?????????? ???????? ??? ????? ????????????? ??? ?????????.

??? ??????????? ?????? XGBoost ?????? ???????????? ????????? ???????? ?????????:

1. **max_depth**: ??????? ??????. ???? ???????? ????????????, ????????? ???????? ????? ???? ?????? ?????? ? ????????. ??????? ??????? ????? ???????? ? ????????????, ??????? ??? ??????? ??????????? ?????????.

2. **learning_rate**: ???????? ????????. ???? ???????? ?????????? ????? ??????? ?????? ? ????????. ????? ?????? ???????? learning_rate ??????? ???????? ????? ???????? ??? ?????????? ??????? ??????????????????, ?? ????? ???????? ?????????? ??????????? ??????.

3. **n_estimators**: ?????????? ???????? ? ????????. ???? ???????? ?????????, ??????? ???????? ??????? ?????????. ??????? ?????????? ???????? ????? ???????? ?????????????????? ??????, ?? ??????? ??????? ????? ????? ???????? ? ????????????.

4. **subsample**: ???? ????????? ????????, ???????????? ??? ?????????? ??????? ??????. ???? ???????? ???????????? ????????? ?????????? ?????? ??? ??????? ??????. ???????? ?????? 1.0 ????????? ????????? ???????????? ? ???????? ???????????? ??????.

5. **colsample_bytree**: ???? ?????????, ???????????? ??? ?????????? ??????? ??????. ???? ???????? ???????????? ????????? ?????????? ????????? ??? ??????? ??????. ???????? ?????? 1.0 ???????? ????????? ???????????? ? ???????? ???????????? ????????.

6. **min_split_gain**: ??????????? ?????????? ??????? ??????, ??????????? ??? ?????????? ??????????? ? ??????. ???? ???????? ???????????? ????????????? ??????. ??????? ???????? min_split_gain ???????? ????????????? ???????????? ?? ???? ?????????? ?????? ??? ?????????? ???????????.

7. **reg_alpha** ? **reg_lambda**: ????????? L1 ? L2 ?????????????. ??? ???????? ?????????????? ????????? ?????? ? ????????????? ???????????? ????? ?????????? ??????? ?? ??????? ???????? ?????.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# ??????????? ?????? LightGBM
model = lgb.LGBMClassifier()

# ??????????? ????? ?????????? ??? ???????????
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.1, 0.05, 0.01, 0.001],
    'n_estimators': [30, 50, 100]
}

# ???????? ??????? GridSearchCV ??? ??????????? ??????????
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='f1')

# ???????? ?????? ? ?????????????? GridSearchCV
start_time = time.time()
grid_search.fit(X_train, y_train)
lgb_gridsearch_execution_time = time.time() - start_time
print("????? ?????? lgb_gridsearch_execution_time:", lgb_gridsearch_execution_time)

# ?????? ????????? ? ???????? ???????
print("?????? ?????????:", grid_search.best_params_)
print("?????? F1:", grid_search.best_score_)

# ?????? ???????? ?????? ?? ???????? ?????? ??????
model = lgb.LGBMClassifier(**grid_search.best_params_)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
lgb_gridsearch_f1 = f1_score(y_pred, y_test)
print("F1 ?? ???????? ?????? ??????:", lgb_gridsearch_f1)


In [ ]:
from hyperopt import fmin, tpe, hp, Trials

# ??????????? ??????? ??? ???????????
def objective(params):
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return -f1_score(y_test, y_pred)  # ??????? ???????, ????????????? F1-????

# ??????????? ???????????? ?????? ??????????
space = {
    'max_depth': hp.choice('max_depth', [3, 4, 5]),
    'learning_rate': hp.choice('learning_rate', [0.1, 0.01, 0.001]),
    'n_estimators': hp.choice('n_estimators', [30, 50, 100])
}

# ?????????? ???????????
trials = Trials()
start_time = time.time()
best = fmin(objective, space, algo=tpe.suggest, max_evals=10, trials=trials)
lgb_hyperopt_execution_time = time.time() - start_time
print("????? ?????? lgb_hyperopt_execution_time:", lgb_hyperopt_execution_time)

# ?????? ????????? ? ???????? ???????
best_params = {
    'max_depth': [3, 4, 5][best['max_depth']],
    'learning_rate': [0.1, 0.01, 0.001][best['learning_rate']],
    'n_estimators': [30, 50, 100][best['n_estimators']]
}
best_score = -trials.best_trial['result']['loss']
print("?????? ?????????:", best_params)
print("?????? F1-????:", best_score)

# ?????? ???????? ?????? ?? ???????? ?????? ??????
model = lgb.LGBMClassifier(**best_params)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
lgb_hyperopt_f1 = f1_score(y_pred, y_test)
print("F1 ?? ???????? ?????? ??????:", lgb_hyperopt_f1)


In [ ]:
import optuna
import lightgbm as lgb
from sklearn.metrics import f1_score

# ??????????? ??????? ??? ???????????
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1),
        'n_estimators': trial.suggest_categorical('n_estimators', [30, 50, 100])
    }
    
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    
    return f1  # ??????? ???????, ????????????? F1-????

# ???????? ??????? Study ??? ???????????
study = optuna.create_study(direction='maximize')
start_time = time.time()
study.optimize(objective, n_trials=10)
lgb_optuna_execution_time = time.time() - start_time
print("????? ?????? lgb_optuna_execution_time:", lgb_optuna_execution_time)

# ?????? ????????? ? ???????? ???????
best_params = study.best_params
best_score = study.best_value

print("?????? ?????????:", best_params)
print("?????? F1-????:", best_score)

# ?????? ???????? ?????? ?? ???????? ?????? ??????
model = lgb.LGBMClassifier(**best_params)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
lgb_optuna_f1 = f1_score(y_pred, y_test)
print("F1 ?? ???????? ?????? ??????:", lgb_optuna_f1)


In [ ]:
from bayes_opt import BayesianOptimization

# ??????????? ??????? ??? ???????????
def objective(max_depth, learning_rate, n_estimators):
    params = {
        'max_depth': int(max_depth),
        'learning_rate': learning_rate,
        'n_estimators': int(n_estimators)
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    
    return f1

# ??????????? ????????? ???????? ?????????? ??? ???????????
pbounds = {
    'max_depth': (3, 5),
    'learning_rate': (0.001, 0.1),
    'n_estimators': (30, 100)
}

# ???????? ??????? BayesianOptimization ??? ???????????
optimizer = BayesianOptimization(
    f=objective,
    pbounds=pbounds,
    random_state=42,
    verbose=2
)

# ?????? ???????????
start_time = time.time()
optimizer.maximize(init_points=2, n_iter=10)
lgb_bayes_opt_execution_time = time.time() - start_time
print("????? ?????? lgb_bayes_opt_execution_time:", lgb_bayes_opt_execution_time)

# ?????? ????????? ? ???????? ???????
best_params = optimizer.max['params']
best_f1 = optimizer.max['target']

print("?????? ?????????:", best_params)
print("?????? F1-????:", best_f1)

# ?????? ???????? ?????? ?? ???????? ?????? ?????? ? ??????? ???????????
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['max_depth'] = int(best_params['max_depth'])
best_model = lgb.LGBMClassifier(**best_params)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
lgb_bayes_opt_f1 = f1_score(y_pred, y_test)
print("F1 ?? ???????? ?????? ??????:", lgb_bayes_opt_f1)

## Catboost

???????? ????????? CatBoost:

1. ????????????? ??????:
   - ???????????????? ?????? CatBoost ? ???????? ?????????????? (????????, ??????? ????????? ??????? ?????????? ??? ?????? ????????? ??? ?????????? ????????? ?????? ??? ?????? ?????????????).

2. ?????????? ??????? ??????? (???????? ???????):
   - CatBoost ?????????? ???????? ???????????? ???????? ??? ????????? ?????????.
   - ??????? ???????? ????????, ?????????? XGBoost ? LightGBM. ?? ?????? ????? ??????????? ????? ?????? ? ?????? ????????, ?????????? ????? ?????????? ????????.
   - CatBoost ????????? ?????? ?????? ? ??????????? ?????????????? ?????????, ?????????? ???????????? ???????? ????????????, ??????? ????????? ?????????????? ????? ??????????? ? ??????????.

3. ?????????? ?????????? ? ?????????? ????????????:
   - ????? ?????????? ??????? ?????? ??????????? ????????? ?????? ????? ?????????????? ?????? ? ????????? ?????????? ??????? ??????????.
   - ???????????? ?????? ??????????? ????? ?????????? ?????????? ?????? ???????????? ?????? ??????, ?????????? XGBoost ? LightGBM.

4. ????????????? ? ?????????????? ????????????:
   - CatBoost ????? ?????????? ????????? ??????? ????????????? ??? ?????????????? ???????????? ??????.
   - ?? ???????????? L1-????????????? (????????????? ?????) ? L2-????????????? (????????????? ????), ??????? ????????? ?????? ? ??????? ??????, ?????????? XGBoost ? LightGBM.

6. ??????? ??????:
   - CatBoost ???????????? ????????? ??????? ??????, ? ??????????? ?? ???? ?????? (????????????? ??? ?????????). ??? ?????? ???????? ????????????? ????? ???????????? ????????????? ??????? ??????, ? ??? ?????? ????????? - ?????????????????? ?????? (MSE), ?????????? XGBoost ? LightGBM.

7. ??????????????? ????????:
   - CatBoost ????? ?????? ???????? ?? ?????????? ????????, ? ???????????? ??????? ?????? ????????????, ????? ???????? ???????? ???????????? ??????.
   - CatBoost ????? ????????? ???? ???????? ?? ?????? ?? ?????? ? ??????????, ?????????? XGBoost ? LightGBM.

8. ???????????????:
   - ????? ???????? ?????? CatBoost ????? ???????????? ??? ??????????????? ?? ????? ??????. ?????? ????????? ??????? ???????? ? ?????????? ???????? ??? ????? ????????????? ??? ?????????.

??? ??????????? ?????? CatBoost ????? ???????? ???????? ?? ????????? ?????? ?????????:

1. **depth**: ??????? ??????. ???? ???????? ??????????, ????????? ???????? ????? ???? ?????? ??????. ??????? ??????? ????? ???????? ? ????????????, ??????? ??? ??????? ??????????? ?????????.

2. **learning_rate**: ???????? ????????. ???? ???????? ???????????? ????? ??????? ?????? ? ????????. ????? ?????? ???????? learning_rate ??????? ???????? ????? ???????? ??? ?????????? ??????? ??????????????????, ?? ????? ???????? ?????????? ??????????? ??????.

3. **iterations**: ?????????? ???????? ? ????????. ???? ???????? ?????????, ??????? ???????? ??????? ?????????. ??????? ?????????? ???????? ????? ???????? ?????????????????? ??????, ?? ??????? ??????? ????? ????? ???????? ? ????????????.

4. **subsample**: ???? ????????? ????????, ???????????? ??? ?????????? ??????? ??????. ???? ???????? ???????????? ????????? ?????????? ?????? ??? ??????? ??????. ???????? ?????? 1.0 ????????? ????????? ???????????? ? ???????? ???????????? ??????.

5. **colsample_bylevel**: ???? ?????????, ???????????? ??? ?????????? ??????? ?????? ??????. ???? ???????? ???????????? ????????? ?????????? ????????? ??? ??????? ?????? ??????. ???????? ?????? 1.0 ???????? ????????? ???????????? ? ???????? ???????????? ????????.

6. **l2_leaf_reg**: L2 ?????????????. ???? ???????? ???????? ?????????????? ????????? ?????? ? ????????????? ???????????? ????? ?????????? ??????? ?? ??????? ???????? ?????.

7. **random_strength**: ???? ??????????? ??? ????????? ????????. ???? ???????? ???????????? ??????? ??????????? ? ?????? ????????? ??? ?????????? ??????. ???????? ?????? 0 ????????? ???????????, ??? ????? ?????? ????????????? ????????????.

8. **bagging_temperature**: ??????????? ????????. ???? ???????? ???????????? ??????? ??????????? ? ?????? ???????? ??? ???????? ??????? ??????. ???????? ?????? 0 ????????? ???????????, ??? ????? ?????? ????????????? ????????????.

????? ????????, ??? ?????????? ????????? ? ?? ???????? ????? ????????????? ? ??????????? ?? ?????????? ?????? ? ?????? ??????. ????????????? ???????? ???????????? ? ??????????? ????????? ? ?????????????? ?????-????????? ??? ?????????? ????????? ?????????????????? ??????.

In [ ]:
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

# ??????????? ?????? CatBoost
model = CatBoostClassifier(silent=True)

# ??????????? ????? ?????????? ??? ???????????
# param_grid = {
#     'depth': [3, 4, 5],
#     'learning_rate': [0.1, 0.05, 0.01],
#     'n_estimators': [30, 50, 100]
# }
param_grid = {
    'depth': [5],
    'learning_rate': [0.1, 0.05, 0.01],
    'n_estimators': [30, 50, 100]
}

# ???????? ??????? GridSearchCV ??? ??????????? ??????????
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='f1')

# ???????? ?????? ? ?????????????? GridSearchCV
start_time = time.time()
grid_search.fit(X_train, y_train)
catboost_gridsearch_execution_time = time.time() - start_time
print("????? ?????? catboost_gridsearch_execution_time:", catboost_gridsearch_execution_time)

# ?????? ????????? ? ???????? ???????
print("?????? ?????????:", grid_search.best_params_)
print("?????? F1:", grid_search.best_score_)

# ?????? ???????? ?????? ?? ???????? ?????? ??????
model = CatBoostClassifier(silent=True, **grid_search.best_params_)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
catboost_gridsearch_f1 = f1_score(y_pred, y_test)
print("F1 ?? ???????? ?????? ??????:", catboost_gridsearch_f1)


In [ ]:
from hyperopt import fmin, tpe, hp, Trials
from sklearn.metrics import f1_score

# ??????????? ??????? ??? ???????????
def objective(params):
    model = CatBoostClassifier(silent=True, **params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return -f1_score(y_test, y_pred)  # ??????? ???????, ????????????? F1-????

# ??????????? ???????????? ?????? ??????????
space = {
    'depth': hp.choice('depth', [3, 4, 5]),
    'learning_rate': hp.choice('learning_rate', [0.1, 0.01, 0.001]),
    'iterations': hp.choice('iterations', [30, 50, 100])
}

# ?????????? ???????????
trials = Trials()
start_time = time.time()
best = fmin(objective, space, algo=tpe.suggest, max_evals=10, trials=trials)
catboost_hyperopt_execution_time = time.time() - start_time
print("????? ?????? catboost_hyperopt_execution_time:", catboost_hyperopt_execution_time)

# ?????? ????????? ? ???????? ???????
best_params = {
    'depth': [3, 4, 5][best['depth']],
    'learning_rate': [0.1, 0.01, 0.001][best['learning_rate']],
    'iterations': [30, 50, 100][best['iterations']]
}
best_score = -trials.best_trial['result']['loss']

print("?????? ?????????:", best_params)
print("?????? F1-????:", best_score)

# ?????? ???????? ?????? ?? ???????? ?????? ??????
model = CatBoostClassifier(silent=True, **best_params)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
catboost_hyperopt_f1 = f1_score(y_pred, y_test)
print("F1 ?? ???????? ?????? ??????:", catboost_hyperopt_f1)


In [ ]:
import optuna

# ??????????? ??????? ??? ???????????
def objective(trial):
    params = {
        'depth': trial.suggest_int('depth', 3, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1),
        'iterations': trial.suggest_categorical('iterations', [30, 50, 100])
    }
    
    model = CatBoostClassifier(silent=True, **params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    
    return f1  # ??????? ???????, ????????????? F1-????

# ???????? ??????? Study ??? ???????????
study = optuna.create_study(direction='maximize')
start_time = time.time()
study.optimize(objective, n_trials=10)
catboost_optuna_execution_time = time.time() - start_time
print("????? ?????? catboost_hyperopt_execution_time:", catboost_optuna_execution_time)

# ?????? ????????? ? ???????? ???????
best_params = study.best_params
best_score = study.best_value

print("?????? ?????????:", best_params)
print("?????? F1-????:", best_score)

# ?????? ???????? ?????? ?? ???????? ?????? ??????
model = CatBoostClassifier(silent=True, **best_params)
model.fit(X_train, y_train)
catboost_optuna_f1 = f1_score(y_test, model.predict(X_test))
print("F1 ?? ???????? ?????? ??????:", catboost_optuna_f1)


In [ ]:
from catboost import CatBoostClassifier
from bayes_opt import BayesianOptimization
from sklearn.metrics import f1_score

# ??????????? ??????? ??? ???????????
def objective(max_depth, learning_rate, n_estimators):
    params = {
        'max_depth': int(max_depth),
        'learning_rate': learning_rate,
        'n_estimators': int(n_estimators)
    }
    model = CatBoostClassifier(silent=True, **params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)

    return f1

# ??????????? ????????? ???????? ?????????? ??? ???????????
pbounds = {
    'max_depth': (3, 5),
    'learning_rate': (0.001, 0.1),
    'n_estimators': (30, 100)
}

# ???????? ??????? BayesianOptimization ??? ???????????
optimizer = BayesianOptimization(
    f=objective,
    pbounds=pbounds,
    random_state=42,
    verbose=2
)

# ?????? ???????????
start_time = time.time()
optimizer.maximize(init_points=2, n_iter=10)
catboost_bayes_opt_execution_time = time.time() - start_time
print("????? ?????? catboost_bayes_opt_execution_time:", catboost_bayes_opt_execution_time)


# ?????? ????????? ? ???????? ???????
best_params = optimizer.max['params']
best_f1 = optimizer.max['target']

print("?????? ?????????:", best_params)
print("?????? F1-????:", best_f1)

# ?????? ???????? ?????? ? ??????? ??????????? ?? ???????? ?????? ??????
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['max_depth'] = int(best_params['max_depth'])
best_model = CatBoostClassifier(silent=True, **best_params)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
catboost_bayes_opt_f1 = f1_score(y_test, y_pred)
print("F1 ?? ???????? ?????? ??????:", catboost_bayes_opt_f1)

In [ ]:
# ???????? ?????? ??? ??????? ????????? ?? f1_score ? ??????? ??????
params_match = [
    ['XGBoost gridsearch', f'{xgb_gridsearch_f1}', f'{xgb_gridsearch_execution_time}'],
    ['XGBoost hyperopt', f'{xgb_hyperopt_f1}', f'{xgb_hyperopt_execution_time}'],
    ['XGBoost optuna', f'{xgb_optuna_f1}', f'{xgb_optuna_execution_time}'],
    ['XGBoost bayes_opt', f'{xgb_bayes_opt_f1}', f'{xgb_bayes_opt_execution_time}'],
    ['LightGBM gridsearch', f'{lgb_gridsearch_f1}', f'{lgb_gridsearch_execution_time}'],
    ['LightGBM hyperopt', f'{lgb_hyperopt_f1}', f'{lgb_hyperopt_execution_time}'],
    ['LightGBM optuna', f'{lgb_optuna_f1}', f'{lgb_optuna_execution_time}'],
    ['LightGBM bayes_opt', f'{lgb_bayes_opt_f1}', f'{lgb_bayes_opt_execution_time}'],
    ['Catboost gridsearch', f'{catboost_gridsearch_f1}', f'{catboost_gridsearch_execution_time}'],
    ['Catboost hyperopt', f'{catboost_hyperopt_f1}', f'{catboost_hyperopt_execution_time}'],
    ['Catboost optuna', f'{catboost_optuna_f1}', f'{catboost_optuna_execution_time}'],
    ['Catboost bayes_opt', f'{catboost_bayes_opt_f1}', f'{catboost_bayes_opt_execution_time}'],

]

# ???????? DataFrame ?? ??????
df = pd.DataFrame(params_match, columns=['?????? ? ??????????', 'F1', '?????'])
pd.set_option('display.max_colwidth', None)
# ??????????? DataFrame
df.head(len(params_match))

In [ ]:
# ???????? ?????? ??? ???????
params_match = [
    ['???????? ???????? (Learning Rate)', 'eta (learning_rate)', 'learning_rate', 'learning_rate'],
    ['?????????? ???????? (Number of Trees)', 'n_estimators', 'n_estimators', 'iterations'],
    ['???????????? ??????? ?????? (Maximum Depth)', 'max_depth', 'max_depth', 'max_depth'],
    ['??????????? ??? ????? ??????, ??????????? ?????????? ???????? ? ????? (Minimum Child Weight)', 'min_child_weight', 'min_child_samples', 'min_data_in_leaf'],
    ['????? (Gamma)', 'gamma', 'min_split_gain', 'l2_leaf_reg'],
    ['???? ??????? ???????? (Subsample Ratio)', 'subsample', 'subsample', 'subsample'],
    ['???? ??????? ????????? (Column Subsampling Ratio)', 'colsample_bytree', 'colsample_bytree', 'colsample_bylevel'],
    ['?????? (L2-?????????????)', 'reg_lambda', 'reg_lambda', 'reg_lambda'],
    ['????? (L1-?????????????)', 'reg_alpha', 'reg_alpha', 'reg_alpha'],
    ['???????????? ????????? ???? (Maximum Delta Step)', 'max_delta_step', '-', '-'],
    ['???????????? ?????????? ?????? (Maximum Bin Count)', 'max_bin', 'max_bin', 'max_bin'],
    ['??????? ????????????? ???????? (Sampling Frequency for LGBM)', '-', 'subsample_freq', '-'],
    ['??????????????? ???? ?????????????? ?????? (Scale the Weight of Positive Class)', 'scale_pos_weight', 'scale_pos_weight', 'scale_pos_weight'],
    ['????? ???????? ? ??????? ???????????? ?????? (Booster Type)', 'booster', 'boosting_type', 'boosting_type'],
    ['??????????? ?????? (Subsample)', 'subsample', 'subsample', 'subsample'],
    ['?????? ??????? ??? ?????????? ?????? (Bagging Fraction)', 'colsample_bytree', 'colsample_bytree', 'colsample_bylevel'],
    ['?????? ?????? ????????? (Split Criterion)', 'tree_method', 'tree_learner', 'grow_policy'],
    ['??????? ?????? (Loss Function)', 'objective', 'objective', 'loss_function'],
    ['???????????? ?????????? ??????? (Max Leaf Nodes)', '-', 'num_leaves', 'max_leaves']
]

# ???????? DataFrame ?? ??????
df = pd.DataFrame(params_match, columns=['???????? ?????????', 'XGBoost', 'LightGBM', 'CatBoost'])
pd.set_option('display.max_colwidth', None)
# ??????????? DataFrame
df
